# 16 - Paired batch: the same seeds flown from the real Checkpoint (125 m) (Kaggle)

Flies seeds 0-149 of the faithful-parameter experiment (random aim points, detector v2) from **125 m over the wide terrain** (`--wide`; see `docs/descent_extension.md`), to compare with the same seeds flown from 45 m (notebook 07). Seeds share sun, aim point, onboard-model noise and start offsets, so differences are paired (`scripts/compare_start_alt.py`). About 200 s per landing; two flight processes run side by side (seeds 0-74 and 75-149), so about 4 to 5 hours.

Settings: **GPU T4 x2**, **Internet on**, add the private dataset `touchdown-detector-v2`. Save Version, Save & Run All (Commit). The runner stops itself at 6 h and keeps finished landings.

In [ ]:
!git clone -q https://github.com/aryanyaksh-art/touch_down.git /tmp/touch_down || (cd /tmp/touch_down && git pull -q)
!sh /tmp/touch_down/scripts/kaggle_setup.sh 2>&1 | tail -5

In [ ]:
import os
# the v2 detector was uploaded as two parts; join them back
!find /kaggle/input -name "best.pt.part*" | sort | xargs cat > /tmp/best.pt
assert os.path.getsize('/tmp/best.pt') == 17520863, os.path.getsize('/tmp/best.pt')

In [ ]:
# the 195 MB global model, then the wide terrain (writes /tmp/data/nightingale_wide_5cm.npz, about 15 s)
!cd /tmp/touch_down && TOUCHDOWN_DATA=/tmp/data python scripts/fetch_data.py --global
!cd /tmp/touch_down && TOUCHDOWN_DATA=/tmp/data python scripts/build_wide_dtm.py --out-dir /tmp/data

In [ ]:
import subprocess
W = '/kaggle/working/wide150'
run = 'python /tmp/touch_down/scripts/kaggle_run.py --ckpt /tmp/best.pt --max-hours 6 --wide'
procs = [subprocess.Popen(f"{run} --seeds {a} {b} --out {W}/part{i}", shell=True) for i, (a, b) in enumerate(((0, 75), (75, 150)))]
for p in procs:
    p.wait()
print('both flight processes finished', [p.returncode for p in procs])

In [ ]:
# merge the two halves into one folder and recompute the summary over all landings
!mkdir -p {W}/all && cp {W}/part0/landing_*.json {W}/part1/landing_*.json {W}/all/
!cd /tmp/touch_down && TOUCHDOWN_DATA=/tmp/data python -m touchdown.analysis.compare_published {W}/all --out {W}/all/summary.json | tail -5
!ls {W}/all | wc -l